# RegD quick start

RegD turns every node of an is-a graph into a box or a ball, so that "x is-a y" becomes "the region of x lies
inside the region of y". This notebook installs the `regd` package, trains it on a toy hierarchy and on the WordNet
mammals, and shows how to query, check, draw and save a model.

It runs on CPU in a minute or two.

## 1. Install

The clone also brings the example data used in section 3.

In [ ]:
!git clone -q --depth 1 https://github.com/HuiYang1997/RegD
!pip install -q ./RegD

## 2. A toy hierarchy

The input is a list of `(child, parent)` pairs. `fit` trains the regions.

In [ ]:
from regd import RegD

edges = [
    ("dog", "mammal"), ("cat", "mammal"), ("whale", "mammal"),
    ("sparrow", "bird"), ("eagle", "bird"),
    ("mammal", "animal"), ("bird", "animal"),
    ("oak", "tree"), ("pine", "tree"), ("tree", "plant"),
    ("animal", "living thing"), ("plant", "living thing"),
]
model = RegD(dim=2, region="box").fit(edges, verbose=False)

`is_a` answers a single question. Note that ("dog", "animal") is not one of the edges: it follows from two of them.

In [ ]:
for child, parent in [("dog", "animal"), ("dog", "living thing"), ("dog", "bird"), ("animal", "dog"), ("oak", "animal")]:
    print(f"{child} is-a {parent}: {model.is_a(child, parent)}   (score {model.score(child, parent):+.3f})")

A pair is accepted when its score is at most `model.tau`. `ancestors` and `descendants` list everything that is accepted, best first.

In [ ]:
print("ancestors of dog:     ", [name for name, score in model.ancestors("dog")])
print("descendants of animal:", [name for name, score in model.descendants("animal")])

In two dimensions the regions can be drawn directly: a child box lies inside its parent box.

In [ ]:
from regd.plot import draw
draw(model, labels=20);

## 3. A real hierarchy: the WordNet mammals

1,179 nodes with the train / validation / test split of the paper. The training file holds the 1,176 direct
edges; the validation and test pairs are pairs that follow from them.

In [ ]:
from regd.benchmarks import load_split, load_vocab

prefix = "RegD/paper/data_utils/data/maxn/mammal_closure.tsv"
names = load_vocab(prefix)                                   # id -> WordNet name
data = {k: [(names[c], names[p]) for c, p in pairs] for k, pairs in load_split(prefix).items()}
print({k: len(v) for k, v in data.items()})
print(data["train"][:3])

WordNet is tree-like, so we use the `strong-bce` preset (see the README for the three presets). Training prints the
validation F1 every 20 epochs and keeps the best checkpoint.

In [ ]:
mammals = RegD(dim=5, region="box", preset="strong-bce")
mammals.fit(data["train"], valid=(data["valid"], data["valid_neg"]),
            nodes=[n for k in ("test", "test_neg") for pair in data[k] for n in pair])

Test F1, and the share of training edges whose child box really lies inside the parent box.

In [ ]:
print("test:       ", mammals.evaluate(data["test"], data["test_neg"]))
print("containment:", round(mammals.containment(), 3))

Predicted ancestors of three nodes against the true ones. Most are found; some are missed or wrong, as the test F1 says.

In [ ]:
# the true ancestors come from the full closure file (name pairs: child, ancestor)
truth = {}
for line in open(prefix):
    child, ancestor = line.split()[:2]
    if child != ancestor and ancestor in mammals.hierarchy.index:
        truth.setdefault(child, set()).add(ancestor)

for node in ["bengal_tiger.n.01", "blue_whale.n.01", "dog.n.01"]:
    predicted = {name for name, score in mammals.ancestors(node)}
    print(node)
    print("   found:  ", sorted(predicted & truth[node]))
    print("   missed: ", sorted(truth[node] - predicted))
    print("   wrong:  ", sorted(predicted - truth[node]))

## 4. Save, load, export

In [ ]:
mammals.save("mammals.pt")
again = RegD.load("mammals.pt")
print(again.is_a("poodle.n.01", "carnivore.n.01"), again.summary())

mammals.export("mammals.tsv")          # one node per line: name, centre, offsets
!head -n 2 mammals.tsv

## 5. Your own graph

Put one `child<TAB>parent` pair per line in a file and pass its path (or a list of pairs, or a `networkx.DiGraph`):

```python
model = RegD(dim=5, region="box").fit("my_edges.tsv")
```

The same from a shell:

```bash
regd train my_edges.tsv --dim 5 --region box --out model.pt
regd query model.pt some_node
```

Things to try: `region="ball"`, another `dim`, another `preset`, or several weights of the depth term at once with
`lam_set=[0, 0.5]` (one training run per value; the best on validation data is kept). All options are listed in
[docs/options.md](https://github.com/HuiYang1997/RegD/blob/main/docs/options.md).